# Exercise set 8

The goals of the exercise are to:
1. Use convolution to remove noise from signals and compute the derivative of noisy signals.
2. Use the Fourier transform to investigate the frequency contributions to signals.

**Learning Objectives:**

After completing this exercise set, you will be able to:

- Smooth noisy signals, and compute their derivatives.
- Use the Fourier transform to find frequency contributions.

**To get the exercise approved, complete the following problems:**

- [8.1(a)](#8.1(a)) and [8.1(b)](#8.1(b)): To show that you can smooth signals and compute derivatives with the Savitzky&ndash;Golay filter.
- [8.2(a)](#8.2(a)): To show that you can use the Fourier transform to find frequencies.

**Files required for this exercise:**
* [Exercise 8.1](#Exercise-8.1): [signal_noise.txt](signal_noise.txt)
* [Exercise 8.2](#Exercise-8.2): [unknown_signal.txt](unknown_signal.txt)
* [Exercise 8.3](#Exercise-8.3): [whitney.wav](whitney.wav) and [emotions.wav](emotions.wav) (only available via Blackboard).

Please ensure that these files are saved in the same directory as this notebook.

## Exercise 8.1

In this exercise, we will remove noise from a synthetic signal. The signal was generated from the following
analytical function,

\begin{equation}
y(t) = \sin (8t) - 1.8t^2 + 0.5t^3.
\label{eq:signal}
\tag{1}\end{equation}

and noise has been added to the signal. The signal can be found in the file [signal_noise.txt](signal_noise.txt), where the first column is the time $t$ and the second column is the signal $y(t)$. The raw data can be loaded as follows:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set_theme(style="ticks", context="notebook", palette="muted")

In [ ]:
data = np.loadtxt("signal_noise.txt")
time, signal = data[:, 0], data[:, 1]
fig, ax = plt.subplots(constrained_layout=True)
ax.plot(time, signal)
ax.set_title("Signal with noise", loc="left")
ax.set(xlabel="Time (t)", ylabel="Signal (y(t))")
sns.despine(fig=fig)

### 8.1(a)

**Task: Apply a Savitzky&ndash;Golay smoothing filter to reduce the noise and plot the smoothed signal, the synthetic signal without noise, and the synthetic signal with noise. What settings did you use for the Savitzky&ndash;Golay filtering to reduce the noise?**

**Note:** You can experiment with the window size and polynomial order for the Savitzky&ndash;Golay filtering to see how the signal is influenced.

**Hint:** The code below shows how you can apply the Savitzky&ndash;Golay filter.

In [ ]:
from scipy.signal import savgol_filter

signal_filter = savgol_filter(signal, window_length=5, polyorder=3)
fig, ax = plt.subplots(constrained_layout=True)
ax.plot(time, signal, label="Signal with noise")
ax.plot(time, signal_filter, label="Filtered signal")
ax.set(xlabel="Time (t)", ylabel="Signal (y(t))")
ax.legend()
sns.despine(fig=fig)

In [ ]:
# Your code here

#### Your answer to question 8.1(a): What settings did you use for the filtering?
*Double click here*

### 8.1(b)

**Task: Apply a Savitzky&ndash;Golay smoothing filter to compute the derivative of the signal with noise. Compare
the computed derivative to the analytical derivative. What settings did you use for the Savitzky&ndash;Golay filtering to get an approximate derivative?**

**Note:** You can experiment with the window size and polynomial order for the Savitzky&ndash;Golay filtering to see how the derivative is influenced.

**Hint:** The code below shows how you can apply the Savitzky&ndash;Golay filter (we use two "extra" arguments, `delta` and `deriv`, compared to the previous example).

In [ ]:
from scipy.signal import savgol_filter

# We need the time step size to get the correct size of
# the derivative:

delta_t = time[1] - time[0]

signal_filter = savgol_filter(
    signal,
    window_length=21,
    polyorder=3,
    delta=delta_t,
    deriv=1,  # Compute the first derivative
)


danalytical = 8 * np.cos(8 * time) - 1.8 * 2 * time + 0.5 * 3 * time**2

fig, ax = plt.subplots(constrained_layout=True)
ax.plot(time, danalytical, label="Derivative (analytical)")
ax.plot(time, signal_filter, label="Derivative (Savitzky–Golay)")
ax.set(xlabel="Time (t)", ylabel="Signal")
ax.legend()
sns.despine(fig=fig)

In [ ]:
# Your code here

#### Your answer to question 8.1(b): What settings did you use for the filtering?
*Double click here*

### 8.1(c)

**Task: Use convolution to smooth the signal. Experiment with the window size and shape. Are you able to remove most of the noise, and what setup did you use?**

**Hint:** Adapt the code below to apply the convolution. There are many options for the [windowing function in SciPy](https://docs.scipy.org/doc/scipy/reference/signal.windows.html), and the code above suggests four you can consider.

In [ ]:
from scipy.signal import get_window

fig, ax = plt.subplots(constrained_layout=True)
ax.plot(time, signal, label="Original signal with noise")

names = ["boxcar", "triang", "hamming", "bartlett"]
for name in names:
    window = get_window(name, 3)  # Get a windowing function of window size 3.
    window /= window.sum()  # Normalise it
    # Run the convolution:
    conv = np.convolve(signal, window, mode="same")
    ax.plot(time, conv, label=f"Filtered signal (window: {name})")
ax.legend()
sns.despine(fig=fig)

In [ ]:
# Your code here

#### Your answer to question 8.1(c): What setup did you use to remove the noise?
*Double click here*

## Exercise 8.2

We will, in this exercise, use the Fourier transform to analyse a signal. In particular, to find what frequencies
are present in the signal.

For performing the Fourier transform, we will make use of the
[`scipy.fft`](https://docs.scipy.org/doc/scipy/reference/fft.html) 
module which contains several methods for applying the **Fast Fourier transform (FFT)**.

Below, you will find Python code for
calculating the FFT for a given signal.

In [ ]:
from scipy.fft import fft, fftfreq


def run_fft(signal, time=None, sampling_rate=None):
    """Apply the Fourier transform for the given signal.

    Parameters
    ----------
    signal : numpy.array
        The measured signal.
    time : numpy.array
        The times we have measured the signal at.
    sampling_rate : float
        How often the signal is sampled per second.

    Returns
    -------
    freq : numpy.array
        The positive frequencies.
    fourier : numpy.array
        The Fourier transform coefficients.
    amplitude : numpy.array
        The amplitude spectrum.
    power : numpy.array
        The power spectrum.

    """
    fourier = fft(signal)
    N = signal.size

    delta_t = None
    if sampling_rate is not None:
        delta_t = 1.0 / sampling_rate
    elif time is not None:
        delta_t = time[1] - time[0]

    if delta_t is not None:
        freq = fftfreq(N, delta_t)
    else:
        freq = np.arange(N)

    amplitude = np.abs(2.0 * fourier / N)

    power = np.abs(fourier) ** 2

    # Normalise power:
    power_sum = power.sum()
    if power_sum > 0:
        power = power / power_sum

    return freq[: N // 2], fourier, amplitude[: N // 2], power[: N // 2]

A few comments about this code:
1. You can supply the time as an array, or the sampling rate (how often the signal is sampled in time). This is
   used to compute the frequencies.
2. The amplitude is computed as `amplitude = np.abs(2.0 * fourier / N`, and we return only the positive half of the spectrum (`freq[:N // 2]`). The FFT will compute both negative and positive frequencies, but for a real signal, the Fourier coefficients for two frequencies $f$ and $-f$ are complex conjugates. The total contribution from these two is the sum $$c_{f} \text{e}^{2\pi i f t } + c_{-f} \text{e}^{-2 \pi i f t},$$ where $c_f$ and $c_{-f}$ are the two coefficients. Due to the complex conjugation (you can show this with [Euler's formula](https://en.wikipedia.org/wiki/Euler\%27s\_formula)), the summation simplifies to a real sinusoid with a peak amplitude of $2 |c_f |$. This means that we can consider the positive frequencies only (done by `N // 2` ) and double the amplitude for our plotting (the factor `2.0` in `amplitude = np.abs(2.0 * fourier / N`).

Here is an example of using the code:

In [ ]:
# Make a signal with an amplitude of 0.3 and frequency of 3:
time_example = np.arange(0, 2, 0.01)
signal_example = 0.3 * np.sin(2.0 * 3 * np.pi * time_example)

# Calculate the Fourier coefficients and amplitude:
freq, _, amplitude, _ = run_fft(signal_example, time=time_example)

# Plot the results
fig, (ax1, ax2) = plt.subplots(
    constrained_layout=True, ncols=2, figsize=(8, 4)
)
ax1.plot(time_example, signal_example, lw=3)
ax2.plot(freq, amplitude, lw=3)
ax1.set(xlabel="Time (s)", ylabel="Signal amplitude")
ax2.set(xlabel="Frequency (Hz)", ylabel="Amplitude")
ax2.axvline(x=3, color="red", ls=":")
ax2.set_xticks(np.arange(0, 21 + 3, 3))
ax2.set_xlim(0, 21)
sns.despine(fig=fig)

### 8.2(a)

The file [unknown_signal.txt](unknown_signal.txt) contains a measurement
of a signal with noise. The first column is the times for which the signal was
measured, and the signal itself can be found in the second column. This
signal, $y$, should have the following form as a function of time, $t$,

\begin{equation}
y(t) = a \sin (f_1 \times 2\pi t) + b \sin(f_2 \times 2\pi t),
\label{eq:signal3}\tag{3}
\end{equation}

where $a$ and $b$ are coefficients, and $f_1$ and $f_2$ are two
frequencies.


**Task: Determine the coefficients and the two frequencies by
calculating the FFT of the given signal.**

In [ ]:
# To load the data do:
data_x = np.loadtxt("unknown_signal.txt")
time_x = data_x[:, 0]
signal_x = data_x[:, 1]

In [ ]:
# Your code here

#### Your answer to question 8.2(a): What are the coefficients ($a$ and $b$) and the frequencies ($f_1$ and $f_2$)?
*Double click here*

## Exercise 8.3

### 8.3(a)

In this exercise, we will determine the frequency of the sustained note [Whitney Houston](https://en.wikipedia.org/wiki/Whitney_Houston) performs in her cover of [I will always love you](https://en.wikipedia.org/wiki/I_Will_Always_Love_You_(Whitney_Houston_recording)). We will do this by analysing a short segment (around 5 seconds) which can be found in the file [whitney.wav](whitney.wav). Note: This file is provided for educational purposes and can only be accessed via Blackboard.

**Task: Identify the frequency and musical note of the sustained vocal.**

You can listen to the segment by using:

In [ ]:
from IPython.display import Audio

# Note: You will probably want to turn **down** the volume before pressing play!
Audio("whitney.wav")

**Some hints:**
1. Frequencies below 200 Hz are often dominated by bass and percussion. Focus your analysis on the range above 200 Hz.
2. You can use [this table](https://en.wikipedia.org/wiki/Piano_key_frequencies#List) to map between frequencies and notes.
3. Look for the highest amplitude.
4. Load the sound file into a NumPy array using [librosa](https://librosa.org/) or [SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.wavfile.read.html), see the examples below. librosa has the benefit that it
   can be used to specify an offset (to skip the first part of the recording).

In [ ]:
# Example 1. Loading the file with librosa:
import librosa

signal_m, sampling_rate = librosa.load(
    "whitney.wav",
    offset=0,  # Do not skip.
    sr=None,  # Do not change the sampling rate.
)
# You can change the offset to start later in the recording,
# e.g., setting offset=1 will skip the first second.
# Normalise the signal before analysis:
signal_m = librosa.util.normalize(signal_m)
duration = len(signal_m) / sampling_rate
time = np.linspace(0, duration, len(signal_m))
fig, ax = plt.subplots(constrained_layout=True)
ax.plot(time, signal_m, rasterized=True)
ax.set(xlabel="Time (s)", ylabel="Amplitude")
sns.despine(fig=fig)

In [ ]:
# Example 2. Loading the file with SciPy:
from scipy.io import wavfile

sampling_rate, signal_m = wavfile.read("whitney.wav")
duration = len(signal_m) / sampling_rate
time = np.linspace(0, duration, len(signal_m))

# Normalise the signal before analysis:
signal_m = signal_m / np.max(np.abs(signal_m))

fig, ax = plt.subplots(constrained_layout=True)
ax.plot(time, signal_m, rasterized=True)
ax.set(xlabel="Time (s)", ylabel="Amplitude")
sns.despine(fig=fig)

In [ ]:
# Your code here

#### Your answer to question 8.3(a): What frequency did you identify?
*Double click here*

### 8.3(b)

In this exercise, we will determine the frequency of the high note(s) [Mariah Carey](https://en.wikipedia.org/wiki/Mariah_Carey) hits in her song [Emotions](https://en.wikipedia.org/wiki/Emotions_(Mariah_Carey_song)). We will do this by analysing a short segment (around 4 seconds) which can be found in the file [emotions.wav](emotions.wav). Note: This file is provided for educational purposes and can only be accessed via Blackboard.

**Task: Identify (approximately) the frequency (range) of the highest note.**

You can listen to the segment by using:

In [ ]:
from IPython.display import Audio

Audio("emotions.wav")
# Note: You will probably want to turn **down** the volume before pressing play!

**Some hints:**
1. Focus your analysis on the range above 1000 Hz (we are looking in the [whistle](https://en.wikipedia.org/wiki/Whistle_register) range).
2. You can use [this table](https://en.wikipedia.org/wiki/Piano_key_frequencies#List) to map between frequencies and notes.
3. Mariah Carey sings with a [vibrato](https://en.wikipedia.org/wiki/Vibrato) meaning that the frequency oscillates with time. This may cause the peak(s) to seem spread out across a small range of frequencies when we plot the FFT frequencies.
4. If you want to compare your answer to previous considerations, you can see [this YouTube video](https://youtu.be/fXEB8YgzcvY?si=t9TtOjc8X88j2lXu&t=10)

In [ ]:
# Your code here

#### Your answer to question 8.3(b): What frequency did you identify?
*Double click here*